# LEQs-xAI · Lab 5 — เทรนโมเดลจำแนกโรคใบพืช แล้วนำไปใช้บนสมาร์ทโฟน
**เป้าหมาย:** อัปโหลดรูปใบ → เทรน MobileNetV2 → ประเมินผล → อธิบายผล (xAI) → ส่งออกเป็น TF.js เพื่อใส่ในแอปมือถือ (PWA)

**วิธีใช้:** รันเซลล์ตามลำดับจากบนลงล่าง (Runtime → Run all ได้ แต่ควรอ่านผลทีละเซลล์)
เปลี่ยน Runtime เป็น GPU ได้ที่ Runtime → Change runtime type (ไม่บังคับ ชุดข้อมูลเล็กใช้ CPU ได้)

**รูปแบบชุดข้อมูล:** ไฟล์ zip หนึ่งไฟล์ ภายในมีโฟลเดอร์ย่อยหนึ่งโฟลเดอร์ต่อหนึ่งคลาส เช่น
```
leaf_dataset.zip
 ├─ ใบปกติ/ 001.jpg 002.jpg ...
 ├─ ใบไหม้/ ...
 └─ ใบจุด/ ...
```
ต้องมีอย่างน้อย 2 คลาส คลาสละ 30 รูปขึ้นไป (แนะนำ 80–150)

## 1. ติดตั้งและตั้งค่า

In [ ]:
import os, sys, types
os.environ["TF_USE_LEGACY_KERAS"] = "1"          # ให้ตัวแปลง TF.js ทำงานได้
!pip -q install tf_keras tensorflowjs
# บางเวอร์ชันของ tensorflowjs ต้อง import โมดูลที่ไม่ได้ใช้ ใส่ตัวแทนว่างเพื่อไม่ให้ error
for n in ["tensorflow_decision_forests", "tensorflow_hub", "jax", "jax.experimental", "jax.experimental.jax2tf"]:
    try: __import__(n)
    except Exception: sys.modules.setdefault(n, types.ModuleType(n))
!apt-get -qq install -y fonts-thai-tlwg > /dev/null 2>&1
import matplotlib, matplotlib.font_manager as fm
for f in fm.findSystemFonts():
    if "Loma" in f or "Garuda" in f: fm.fontManager.addfont(f)
matplotlib.rcParams["font.family"] = ["Loma", "Garuda", "DejaVu Sans"]
import numpy as np, tensorflow as tf, json, shutil, zipfile, glob
print("TensorFlow", tf.__version__, "| GPU:", bool(tf.config.list_physical_devices("GPU")))

In [ ]:
# ตั้งค่า (แก้ได้)
IMG = 160                # ขนาดภาพที่เข้าโมเดล
BATCH = 16
EPOCHS_HEAD = 8          # รอบเทรนส่วนหัว
EPOCHS_FINE = 4          # รอบ fine-tune (0 = ข้าม)
THRESHOLD = 0.60         # ต่ำกว่านี้แอปจะแสดง "ไม่แน่ใจ"
WEIGHTS = os.environ.get("WEIGHTS", "imagenet")   # อย่าแก้
DATA_DIR = "/content/dataset"

## 2. อัปโหลดชุดข้อมูล

In [ ]:
if not os.path.isdir(DATA_DIR):
    try:
        from google.colab import files
        up = files.upload()                       # เลือกไฟล์ zip
        z = list(up.keys())[0]
        os.makedirs(DATA_DIR, exist_ok=True)
        zipfile.ZipFile(z).extractall(DATA_DIR)
    except ImportError:
        raise SystemExit("ไม่ได้รันบน Colab: วางโฟลเดอร์ข้อมูลที่ " + DATA_DIR)
# ถ้า zip มีโฟลเดอร์ครอบอีกชั้น ให้ลงไปหนึ่งชั้น
sub = [d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d)) and not d.startswith(("__", "."))]
if len(sub) == 1 and any(os.path.isdir(os.path.join(DATA_DIR, sub[0], x)) for x in os.listdir(os.path.join(DATA_DIR, sub[0]))):
    DATA_DIR = os.path.join(DATA_DIR, sub[0])
classes = sorted(d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d)) and not d.startswith(("__", ".")))
for c in classes:
    print(f"{c}: {len([f for f in os.listdir(os.path.join(DATA_DIR, c)) if f.lower().endswith(('.jpg','.jpeg','.png','.webp','.bmp'))])} รูป")
assert len(classes) >= 2, "ต้องมีอย่างน้อย 2 คลาส"

## 3. แบ่งข้อมูล train / validation และเพิ่มความหลากหลายของภาพ (augmentation)

In [ ]:
common = dict(image_size=(IMG, IMG), batch_size=BATCH, seed=42, validation_split=0.2, label_mode="int")
train_raw = tf.keras.utils.image_dataset_from_directory(DATA_DIR, subset="training", **common)
val_ds = tf.keras.utils.image_dataset_from_directory(DATA_DIR, subset="validation", **common)
class_names = train_raw.class_names
print("คลาส:", class_names)

def augment(x, y):
    x = tf.image.random_flip_left_right(x)
    x = tf.image.random_brightness(x, 25.0)
    x = tf.image.random_contrast(x, 0.8, 1.2)
    return tf.clip_by_value(x, 0.0, 255.0), y

train_ds = train_raw.map(augment, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(tf.data.AUTOTUNE)

## 4. สร้างโมเดล (Transfer Learning จาก MobileNetV2)
ใช้โมเดลที่เรียนรู้ภาพทั่วไปมาแล้ว (ImageNet) แล้วเทรนเฉพาะชั้นสุดท้ายให้จำแนกคลาสของเรา
ชั้น `Rescaling` อยู่ในโมเดล ดังนั้นแอปมือถือป้อนค่าพิกเซล 0–255 ได้ตรง ๆ

In [ ]:
base = tf.keras.applications.MobileNetV2(input_shape=(IMG, IMG, 3), include_top=False,
                                         weights=None if WEIGHTS == "none" else "imagenet", alpha=0.35)
base.trainable = False
model = tf.keras.Sequential([
    tf.keras.layers.Rescaling(1/127.5, offset=-1, input_shape=(IMG, IMG, 3)),
    base,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(len(class_names), activation="softmax"),
])
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
print("พารามิเตอร์ทั้งหมด:", model.count_params())

## 5. เทรน

In [ ]:
h1 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD, verbose=2)
hist = {k: list(v) for k, v in h1.history.items()}
if EPOCHS_FINE > 0:
    base.trainable = True
    for l in base.layers[:-30]: l.trainable = False      # ปลดล็อกเฉพาะชั้นท้าย
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    h2 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FINE, verbose=2)
    for k, v in h2.history.items(): hist[k] += list(v)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(hist["accuracy"], label="train"); ax[0].plot(hist["val_accuracy"], label="val"); ax[0].set_title("Accuracy"); ax[0].legend()
ax[1].plot(hist["loss"], label="train"); ax[1].plot(hist["val_loss"], label="val"); ax[1].set_title("Loss"); ax[1].legend()
plt.show()

## 6. ประเมินผล
ดู confusion matrix และ accuracy ต่อคลาส ถ้า train สูงแต่ val ต่ำมาก แปลว่า overfit (ข้อมูลน้อยหรือหน้าตาซ้ำกันเกินไป)

In [ ]:
y_true, y_pred = [], []
for x, y in val_ds:
    y_true += list(y.numpy()); y_pred += list(np.argmax(model.predict(x, verbose=0), axis=1))
y_true, y_pred = np.array(y_true), np.array(y_pred)
n = len(class_names)
cm = np.zeros((n, n), int)
for t, p in zip(y_true, y_pred): cm[t, p] += 1
print("Accuracy (validation):", round(float((y_true == y_pred).mean()), 3), "จาก", len(y_true), "รูป")
for i, c in enumerate(class_names):
    tot = cm[i].sum(); print(f"  {c}: ถูก {cm[i,i]}/{tot}")
plt.figure(figsize=(4.5, 4)); plt.imshow(cm, cmap="Blues")
plt.xticks(range(n), class_names, rotation=45); plt.yticks(range(n), class_names)
for i in range(n):
    for j in range(n): plt.text(j, i, cm[i, j], ha="center", va="center")
plt.xlabel("ทำนาย"); plt.ylabel("จริง"); plt.tight_layout(); plt.show()

## 7. อธิบายผล (xAI) ด้วย Occlusion
ปิดภาพทีละช่องแล้วดูว่าความมั่นใจลดลงเท่าใด ช่องที่ทำให้ความมั่นใจลดมากคือบริเวณที่โมเดลใช้ตัดสินใจ
ตรวจว่าโมเดลมองที่ใบ ไม่ใช่พื้นหลังหรือมือ

In [ ]:
def occlusion_map(img, cls, patch=32, stride=16):
    base_p = model.predict(img[None], verbose=0)[0][cls]
    H = (IMG - patch) // stride + 1
    heat = np.zeros((H, H))
    for i in range(H):
        for j in range(H):
            x = img.copy(); x[i*stride:i*stride+patch, j*stride:j*stride+patch, :] = 127.0
            heat[i, j] = base_p - model.predict(x[None], verbose=0)[0][cls]
    return base_p, heat

xb, yb = next(iter(val_ds))
show = min(4, len(xb))
fig, ax = plt.subplots(2, show, figsize=(3*show, 6), squeeze=False)
for k in range(show):
    img = xb[k].numpy(); pred = int(np.argmax(model.predict(img[None], verbose=0)[0]))
    p, heat = occlusion_map(img, pred)
    ax[0, k].imshow(img.astype("uint8")); ax[0, k].set_title(f"{class_names[pred]} {p:.0%}"); ax[0, k].axis("off")
    ax[1, k].imshow(img.astype("uint8")); ax[1, k].imshow(heat, cmap="jet", alpha=0.5, extent=(0, IMG, IMG, 0)); ax[1, k].axis("off")
plt.tight_layout(); plt.show()

## 8. ส่งออกโมเดลเป็น TF.js สำหรับแอปมือถือ
ผลลัพธ์: โฟลเดอร์ `model/` (model.json + ไฟล์น้ำหนัก) และ `labels.json` รวมเป็น `leaf_model_for_app.zip`
นำไปวางในโฟลเดอร์แอป PWA ตามคู่มือ Lab 5B โดยแทนที่ `model/` และ `labels.json` เดิม

In [ ]:
import tensorflowjs as tfjs
OUT = "/content/export"
shutil.rmtree(OUT, ignore_errors=True); os.makedirs(OUT + "/model")
tfjs.converters.save_keras_model(model, OUT + "/model")
json.dump({"input_size": IMG, "threshold": THRESHOLD, "labels": class_names},
          open(OUT + "/labels.json", "w"), ensure_ascii=False)
# TFLite สำหรับแอป Flutter/Android (ทางเลือก)
open(OUT + "/model.tflite", "wb").write(tf.lite.TFLiteConverter.from_keras_model(model).convert())
shutil.make_archive("/content/leaf_model_for_app", "zip", OUT)
for f in sorted(glob.glob(OUT + "/**/*", recursive=True)):
    if os.path.isfile(f): print(f"{os.path.getsize(f)/1024:8.1f} KB  {os.path.relpath(f, OUT)}")

In [ ]:
try:
    from google.colab import files
    files.download("/content/leaf_model_for_app.zip")
except ImportError:
    print("ไฟล์อยู่ที่ /content/leaf_model_for_app.zip")

## 9. ทดสอบก่อนนำไปใช้ (checklist)
- accuracy ของ validation สูงกว่า 80% และไม่มีคลาสใดผิดเกินครึ่ง
- ภาพ occlusion ชี้ที่ใบ ไม่ใช่พื้นหลัง
- ถ่ายใบจริงด้วยมือถือ 5 รูปต่อคลาสที่โมเดลไม่เคยเห็น แล้วนับว่าตอบถูกกี่รูป
- ถ่ายสิ่งที่ไม่ใช่ใบ (มือ โต๊ะ) แล้วดูว่าแอปแสดง "ไม่แน่ใจ" หรือไม่